<a href="https://colab.research.google.com/github/Rahat048/Batch-62/blob/main/LLM_Models_for_Creative_Video_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade fal-client

In [2]:
from google.colab import userdata
LIGHTRICK_API = userdata.get('LIGHTRICK_API')

In [3]:
import os
os.environ["LIGHTRICK_API"] = LIGHTRICK_API

In [4]:
from fal_client import SyncClient

In [5]:
sync_client = SyncClient(LIGHTRICK_API)

In [6]:
run = sync_client.run(
    "fal-ai/ltx-video",    arguments={
        "prompt": "A man stands waist-deep in a crystal-clear mountain pool, his back turned to a massive, thundering waterfall that cascades down jagged cliffs behind him. He wears a dark blue swimming shorts and his muscular back glistens with water droplets. The camera moves in a dynamic circular motion around him, starting from his right side and sweeping left, maintaining a slightly low angle that emphasizes the towering height of the waterfall. As the camera moves, the man slowly turns his head to follow its movement, his expression one of awe as he gazes up at the natural wonder. The waterfall creates a misty atmosphere, with sunlight filtering through the spray to create rainbow refractions. The water churns and ripples around him, reflecting the dramatic landscape. The handheld camera movement adds a subtle shake that enhances the raw, untamed energy of the scene. The lighting is natural and bright, with the sun positioned behind the waterfall, creating a backlit effect that silhouettes the falling water and illuminates the mist."
    },
)

In [7]:
print(run)

{'video': {'url': 'https://v3.fal.media/files/zebra/dgyUIUxwjfObdS77ocTz2_tmp7fwlqyy9.mp4', 'content_type': 'application/octet-stream', 'file_name': 'tmp7fwlqyy9.mp4', 'file_size': 7033139}, 'seed': 84742046}


In [8]:
!pip install --upgrade --quiet google_genai

In [9]:
!pip install --quiet gTTS

In [10]:
!pip install playsound

  Preparing metadata (setup.py) ... done
  Created wheel for playsound: filename=playsound-1.3.0-py3-none-any.whl size=7020 sha256=f0ccacca833bedc5a8e7eff7a7f6f0bd8ef4196541a4fa36b9b0e2c05b4e83e1
  Stored in directory: /root/.cache/pip/wheels/90/89/ed/2d643f4226fc8c7c9156fc28abd8051e2d2c0de37ae51ac45c
Successfully built playsound


In [11]:
from google.colab import userdata
GEMINI_API: str = userdata.get('GOOGLE_API_KEY')
if(GEMINI_API):
  print("API Key found")
else:
  print("API Key not found")

API Key found


In [12]:
from google import genai
from google.genai import Client

client: Client = genai.Client(
    api_key=GEMINI_API,
)

model: str = "gemini-2.0-flash-exp"

In [13]:
import time

def upload_video(video_file_name):
  video_file = client.files.upload(path=video_file_name)
  while video_file.state == "PROCESSING":
      print('Waiting for video to be processed.')
      time.sleep(10)
      video_file = client.files.get(name=video_file.name or "")

  if video_file.state == "FAILED":
    raise ValueError(video_file.state)
  print(f'Video processing complete: ' + (video_file.uri or ""))

  return video_file

In [14]:
my_video = upload_video(video_file_name = "/content/UOaO4rim8kmmCnxn8uy7W_tmpdlzfb_ho (1).mp4" )

Waiting for video to be processed.
Video processing complete: https://generativelanguage.googleapis.com/v1beta/files/p0dixm8uupoz


In [15]:
from google.genai.types import Content, Part
from IPython.display import display, Markdown

In [16]:
prompt = """For each scene in this video,
            generate captions that describe the scene along with any spoken text placed in quotation marks.
            Place each caption into an object with the timecode of the caption in the video.
         """

video = my_video
response = client.models.generate_content(
    model=model,
    contents=[
        Content(
            role="user",
            parts=[
                Part.from_uri(
                    file_uri=video.uri or "",
                    mime_type=video.mime_type or ""),
                ]),
        prompt,
    ]
)

Markdown(response.text)

```json
[
  {
    "timecode": "00:00",
    "caption": "A shirtless man stands in a body of water with a large waterfall in the background."
  },
  {
    "timecode": "00:01",
     "caption": "A shirtless man, wearing dark blue swim trunks, faces away from the camera toward the waterfall."
  },
  {
    "timecode": "00:02",
     "caption": "A shirtless man with dark hair stands in the water, looking at a large waterfall."
  },
  {
    "timecode": "00:03",
     "caption": "A shirtless man, with his back to the camera, stands in water in front of a waterfall."
  },
    {
    "timecode": "00:04",
      "caption": "A shirtless man in dark blue swim trunks stands facing a waterfall."
  }
]
```

In [18]:
from gtts import gTTS
from playsound import playsound
from IPython.display import Audio

In [19]:
tts = gTTS(text= response.text, lang='en')

with open('output.mp3', 'wb') as f:
    for chunk in tts.stream():
        f.write(chunk)

In [23]:
display(Audio('output.mp3', autoplay=True))